In [71]:
# Pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Helper libraries
import numpy as np
import matplotlib.pyplot as plt

In [72]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [73]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
          mean = [0.485, 0.456, 0.406],
          std = [0.229, 0.224, 0.225]
    )
])

trainset = torchvision.datasets.CIFAR10(root='./data', train = True, download = True, transform = transform)
testset = torchvision.datasets.CIFAR10(root=',/data', train = False, download = True, transform = transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size = 128, shuffle = True)
testloader = torch.utils.data.DataLoader(testset, batch_size = 128, shuffle = False)

In [74]:
#Basic Block(Plain)
class PlainBasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride = 1):
        super().__init__()

        self.conv1 = nn.Conv2d(
              in_channels, out_channels,
              kernel_size = 3, stride = stride, padding = 1, bias = False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels, out_channels,
            kernel_size = 3, stride = 1, padding = 1, bias = False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        #PlainBasicBlock 이기 때문에 Shortcut 사용안함

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        return out

In [75]:
#Plain ResNet-34
class PlainResNet34(nn.Module):
    def __init__(self, num_classes = 10):
        super().__init__()
        self.in_channels = 64

        self.conv1 = nn.Conv2d(
            3, 64, kernel_size = 3,
            stride = 1, padding = 1, bias = False
        )
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(64, 3, stride =1)
        self.layer2 = self._make_layer(128, 4, stride =2)
        self.layer3 = self._make_layer(256, 6, stride =2)
        self.layer4 = self._make_layer(512, 3, stride =2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, out_channels, blocks, stride):
        layers = []
        layers.append(
            PlainBasicBlock(self.in_channels, out_channels, stride)
        )
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(
                PlainBasicBlock(self.in_channels, out_channels)
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [76]:
#Plain34 모델 작동 확인
model_plain34 = PlainResNet34(num_classes = 10)
dummy = torch.randn(1, 3, 32, 32)
out = model_plain34(dummy)
print(out.shape)

torch.Size([1, 10])


In [77]:
#Basic Block (ResNet-34)
class BasicBlock(nn.Module):
  expansion = 1

  def __init__(self, in_channels, out_channels, stride = 1):
      super(BasicBlock, self).__init__()

      self.conv1 = nn.Conv2d(
          in_channels, out_channels,
          kernel_size = 3, stride = stride, padding = 1, bias = False
      )
      self.bn1 = nn.BatchNorm2d(out_channels)

      self.conv2 = nn.Conv2d(
          out_channels, out_channels,
          kernel_size = 3, stride = 1, padding = 1, bias = False
      )
      self.bn2 = nn.BatchNorm2d(out_channels)

      self.shortcut = nn.Sequential()
      if stride != 1 or in_channels != out_channels:
          self.shortcut = nn.Sequential(
              nn.Conv2d(
                  in_channels, out_channels,
                  kernel_size = 1, stride = stride, bias = False
              ),
              nn.BatchNorm2d(out_channels)
          )

  def forward(self, x):
      out = F.relu(self.bn1(self.conv1(x)))
      out = self.bn2(self.conv2(out))
      out += self.shortcut(x)
      out = F.relu(out)
      return out

In [78]:
#ResNet-34
class ResNet34(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet34, self).__init__()
        self.in_channels = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size =3,
                               stride = 1, padding = 1, bias = False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(64, 3, stride = 1)
        self.layer2 = self._make_layer(128, 4, stride = 2)
        self.layer3 = self._make_layer(256, 6, stride = 2)
        self.layer4 = self._make_layer(512, 3, stride = 2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, out_channels, blocks, stride):
        layers = []
        layers.append(BasicBlock(self.in_channels, out_channels, stride))
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return (x)

In [79]:
#ResNet-34 모델 작동 확인
model = ResNet34(num_classes = 10)
print(model)

dummy = torch.randn(1, 3, 32, 32)
out = model(dummy)
print(out.shape)

ResNet34(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential()
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affin

In [80]:
#Bottleneck Block(Plain)

In [81]:
class PlainBottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, out_channels, stride = 1):
        super().__init__()

        self.conv1 = nn.Conv2d(
              in_channels, out_channels,
              kernel_size = 1, bias = False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels, out_channels,
            kernel_size = 3, stride = stride, padding = 1, bias = False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.conv3 = nn.Conv2d(
              out_channels, out_channels * self.expansion,
              kernel_size = 1, bias = False
        )
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)

        #PlainBottleneckBlock 이기 때문에 Shortcut 사용안함

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = F.relu(self.bn3(self.conv3(out)))
        return out

In [82]:
#Plain ResNet-50

In [83]:
class PlainResNet50(nn.Module):
    def __init__(self, num_classes = 10):
        super().__init__()
        self.in_channels = 64

        self.conv1 = nn.Conv2d(
            3, 64, kernel_size = 3,
            stride = 1, padding = 1, bias = False
        )
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(64, 3, stride = 1)
        self.layer2 = self._make_layer(128, 4, stride = 2)
        self.layer3 = self._make_layer(256, 6, stride = 2)
        self.layer4 = self._make_layer(512, 3, stride = 2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * PlainBottleneck.expansion, num_classes)

    def _make_layer(self, out_channels, blocks, stride):
        layers = []
        layers.append(
            PlainBottleneck(self.in_channels, out_channels, stride)
        )
        self.in_channels = out_channels * PlainBottleneck.expansion

        for _ in range(1, blocks):
            layers.append(
                PlainBottleneck(self.in_channels, out_channels)
            )

        return nn.Sequential(*layers)

    def forward(self, x):
      x = F.relu(self.bn1(self.conv1(x)))
      x = self.layer1(x)
      x = self.layer2(x)
      x = self.layer3(x)
      x = self.layer4(x)
      x = self.avgpool(x)
      x = torch.flatten(x, 1)
      x = self.fc(x)
      return x

In [84]:
#Plain50 모델 작동 확인

In [85]:
model_plain50 = PlainResNet50(num_classes = 10)
dummy = torch.randn(1, 3, 32, 32)
out = model_plain50(dummy)
print(out.shape)

torch.Size([1, 10])


In [86]:
#Bottleneck Block (ResNet-50)
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1):
        super(Bottleneck, self).__init__()

        self.conv1 = nn.Conv2d(
            in_channels, out_channels,
            kernel_size = 1, bias = False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels, out_channels,
            kernel_size = 3, stride = stride, padding =1, bias = False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.conv3 = nn.Conv2d(
            out_channels, out_channels * self.expansion,
            kernel_size = 1, bias = False
        )
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                      in_channels, out_channels * self.expansion,
                      kernel_size = 1, stride = stride, bias = False
                ),
                nn.BatchNorm2d(out_channels * self.expansion)
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

In [87]:
#ResNet-50
class ResNet50(nn.Module):
    def __init__(self, num_classes = 10):
        super(ResNet50, self).__init__()
        self.in_channels = 64

        self.conv1 = nn.Conv2d(
            3, 64, kernel_size = 3,
            stride = 1, padding = 1, bias = False
        )
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(64, 3, stride =1)
        self.layer2 = self._make_layer(128, 4, stride =2)
        self.layer3 = self._make_layer(256, 6, stride =2)
        self.layer4 = self._make_layer(512, 3, stride = 2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * Bottleneck.expansion, num_classes)

    def _make_layer(self, out_channels, blocks, stride):
        layers = []
        layers.append(Bottleneck(self.in_channels, out_channels, stride))
        self.in_channels = out_channels * Bottleneck.expansion

        for _ in range(1, blocks):
            layers.append(Bottleneck(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [88]:
#ResNet-50 모델 작동 확인
model = ResNet50(num_classes = 10)
print(model)

dummy = torch.randn(1, 3, 32, 32)
out = model(dummy)
print(out.shape)

ResNet50(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (1): Bottleneck(
     

In [89]:
def train_and_eval(model, trainloader, testloader, epochs=15):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=0.001,
        momentum=0.9,
        weight_decay=1e-4
    )

    train_losses = []
    val_accuracies = []

    for epoch in range(epochs):
        # Train
        model.train()
        running_loss = 0.0

        for inputs, labels in trainloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(trainloader)
        train_losses.append(avg_loss)

        # Validation
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in testloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        val_acc = 100 * correct / total
        val_accuracies.append(val_acc)

        print(f"[{epoch+1}/{epochs}] "
              f"Loss: {avg_loss:.4f} | "
              f"Val Acc: {val_acc:.2f}%")

    return train_losses, val_accuracies

In [90]:
resnet34 = ResNet34(num_classes=10)
plain34 = PlainResNet34(num_classes=10)

resnet50 = ResNet50(num_classes=10)
plain50 = PlainResNet50(num_classes=10)

In [92]:
# ResNet-34 (residual)
loss_r34, acc_r34 = train_and_eval(resnet34, trainloader, testloader, epochs=5)

# Plain ResNet-34
loss_p34, acc_p34 = train_and_eval(plain34, trainloader, testloader, epochs=5)

# ResNet-50 (residual)
loss_r50, acc_r50 = train_and_eval(resnet50, trainloader, testloader, epochs=5)

# Plain ResNet-50
loss_p50, acc_p50 = train_and_eval(plain50, trainloader, testloader, epochs=5)

KeyboardInterrupt: 

In [93]:
plt.plot(acc_r34, label="ResNet-34 (Residual)")
plt.plot(acc_p34, label="ResNet-34 (Plain)")
plt.plot(acc_r50, label="ResNet-50 (Residual)")
plt.plot(acc_p50, label="ResNet-50 (Plain)")

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.title("Ablation Study: Plain vs Residual ResNet")
plt.legend(loc = "best")
plt.show()

NameError: name 'acc_r34' is not defined

In [94]:
print("Final Validation Accuracy")
print(f"ResNet-34 (Residual): {acc_r34[-1]:.2f}%")
print(f"ResNet-34 (Plain):    {acc_p34[-1]:.2f}%")
print(f"ResNet-50 (Residual): {acc_r50[-1]:.2f}%")
print(f"ResNet-50 (Plain):    {acc_p50[-1]:.2f}%")

Final Validation Accuracy


NameError: name 'acc_r34' is not defined